In [0]:
Monthly Percentage Difference

Given a table of purchases by date, calculate the month-over-month percentage change in revenue. The output should include the year-month date (YYYY-MM) and percentage change, rounded to the 2nd decimal point, and sorted from the beginning of the year to the end of the year.
The percentage change column will be populated from the 2nd month forward and can be calculated as ((this month's revenue - last month's revenue) / last month's revenue)*100.



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder.appName("MonthlyPercentageDiff").getOrCreate()

# Dummy data: id, created_at, value (revenue), purchase_id
data = [
    (1, "2020-01-05", 1000, 101),
    (2, "2020-01-15", 500, 102),
    (3, "2020-01-20", 750, 103),
    (4, "2020-02-10", 1200, 104),
    (5, "2020-02-14", 800, 105),
    (6, "2020-02-18", 600, 106),
    (7, "2020-03-05", 1500, 107),
    (8, "2020-03-12", 900, 108),
    (9, "2020-03-25", 1100, 109),
    (10, "2020-04-03", 800, 110),
    (11, "2020-04-17", 600, 111),
    (12, "2020-05-01", 2000, 112),
    (13, "2020-05-20", 1500, 113),
    (14, "2020-06-10", 1000, 114),
    (15, "2020-06-15", 1200, 115),
]

# Create DataFrame
df = spark.createDataFrame(data, ["id", "created_at", "value", "purchase_id"])
df = df.withColumn("created_at", to_date(col("created_at")))

df.createOrReplaceTempView("sf_transactions")

# Preview
df.show()

In [0]:
spark.sql("""
    with monthly_revenue as (
    select date_format(created_at,'yyyy-MM') as month, sum(value) as revenue
    from sf_transactions
    group by date_format(created_at,'yyyy-MM' )),

    revenue_with_lag as (
    select month, revenue,
    LAG(revenue,1, 0) OVER (ORDER BY month) as prev_month_revenue
    from monthly_revenue)

    select month, 
    round(try_divide((revenue - prev_month_revenue), prev_month_revenue)*100,2) as pct_change
    from revenue_with_lag


""").display()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
revenue_df = df.withColumn('year_month', F.date_format(F.col('created_at'), 'yyyy-MM')) \
    .groupBy(F.col('year_month')) \
    .agg(F.sum(F.col('value')).alias('revenue'))

window_spec = Window.orderBy(F.col('year_month'))
revenue_df = revenue_df.withColumn('prev_month_revenue', F.lag('revenue',1,0).over(window_spec)) \
    .withColumn('pct_change', F.round(F.try_divide(F.col('revenue') - F.col('prev_month_revenue'),F.col('prev_month_revenue')),2))                 
revenue_df.display()